# Dunnhumby seed43 — M2 공유 N/V 규제만 완화
**새 학습은 M2 하나뿐입니다.** 직전 N/V 조건부 ID 표현은 그대로 두고 공유 N/V L2만 `0.001 → 0.0001`로 낮춥니다. 기존 M1·M2(.001)의 정확히 일치하는 결과와 전체 곡선을 재사용합니다. 없거나 조건이 다르면 학습 전에 중단하며, 기준모형을 자동 재학습하지 않습니다. M3·M4·M5, H&M, 추가 시드는 이번 실행에 없습니다.

표현은 사용자·상품 각각 `z̃ = z ⊙ [1 + .05 tanh(A_N φ_N) + .05 tanh(A_V φ_V)]`, 점수는 `z̃_uᵀz̃_i`입니다. N과 V를 모두 유지하고 네 공유행렬과 ID를 하나의 optimizer·추천손실로 공동학습합니다. N은 거래발생/활동이며 동일상품 재구매율이 아닙니다. q_C가 없는 **CLV 구성요소 표현**입니다. 동결·외부 보정·재정렬은 없습니다.

고정: ID64·2층·ID L2=.001·η_N=η_V=.05·lr=.0005·batch8192·binary 그래프·uniform K=1·무가중 BPR. L2는 `.001 × 배치 ID 제곱합/B + .0001 × 공유 N/V 행렬 제곱합`입니다. N/V 입력·RBF·학습 양성의 구매자 기여 제외(LOO)·초기화는 직전 모델과 같습니다. 규제가 실패 원인으로 입증됐다는 뜻은 아닙니다.

학습≤683 / 이미 노출된 개발684–690, 신규상품(학습 user-item 쌍 제외), MIN_ITEM_INTER=1. 최대300 epoch·25마다 평가. 전체 가격·구매금액 가중 적중값@10으로 checkpoint 선택, 동률이면 앞선 epoch, 100 이후4회 미개선 종료, 최적 checkpoint 복원. **최대300이지 300까지 반드시 학습하는 것은 아닙니다.** 최종 test/holdout은 사용하지 않습니다(기존 코드의 `test` 명칭은 개발684–690입니다).

비교: 전체 가격·구매금액 가중 적중값@10 및 가중 NDCG@10의 M1/직전 M2 대비 변화, 여섯 Recall/NDCG 각각 M1의99% 보호. 각자 선택 checkpoint와 같은 epoch 비교를 별도 저장합니다. 전체·저/중/고CLV·@20/@50·노출의 불리한 결과도 모두 보존합니다. 고CLV는 보조이며 단일 개발시드로 유의성·일반화·CLV 귀속을 주장하지 않습니다. 실패해도 추가 계수/시드를 자동 실행하지 않습니다.

로컬 검증: 기존 결과 SHA·소스 일치, CPU 합성 학습·중단/재개·완료 캐시, 두 축 작동, L2만 변경, 전체 표 저장. Drive 연결과 실제 전체 GPU 실험은 아직 수행하지 않았습니다.

In [ ]:
from pathlib import Path
import os, sys, subprocess, json
from google.colab import drive
ROOT = Path('/content/drive/MyDrive/논문/data')
REPORT = ROOT/'results_v3_dunnhumby_nv_modulated_id_seed43_v3/reports/result.json'
# 원본 JSON을 옮겼다면 REPORT만 수정하세요. 기존 nv_modulated_id_m2_m5_seed43_results.zip도 허용됩니다.
try:
    if not ROOT.is_dir():
        drive.mount('/content/drive')
    if not ROOT.is_dir():
        raise RuntimeError('논문/data 폴더가 없습니다. Drive 계정과 ROOT 경로를 확인하세요.')
    if not REPORT.is_file():
        raise RuntimeError(f'직전 결과가 없습니다: {REPORT}. REPORT를 원본 JSON 또는 ZIP 경로로 수정하세요. 새 학습은 시작하지 않았습니다.')
except (OSError, ValueError, NotImplementedError) as exc:
    raise RuntimeError('Drive 연결/읽기 실패. 계정과 마운트 상태를 확인한 뒤 이 셀부터 재실행하세요. 새 학습은 시작하지 않았습니다.') from exc
SOURCE_COMMIT = 'f107127b3c0c68a2636c1cc85868e3ea1056eb5f'
REPO = Path('/content/clv-nv-l2-' + SOURCE_COMMIT[:12])
if not REPO.exists():
    subprocess.run(['git', 'clone', 'https://github.com/jung-un/clv-m2-lightgcn-runner.git', str(REPO)], check=True)
subprocess.run(['git', '-C', str(REPO), 'checkout', '--detach', SOURCE_COMMIT], check=True)
assert subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip() == SOURCE_COMMIT
if 'lightgcn_clv_v3' in sys.modules:
    assert Path(sys.modules['lightgcn_clv_v3'].__file__).resolve().parent == REPO.resolve(), '다른 실험 코드가 로드되어 있습니다. 런타임을 다시 시작하세요.'
os.chdir(REPO)
sys.path.insert(0, str(REPO))
import clv_nv_modulated_l2_screen as screen
import pandas as pd
OUT = ROOT/'results_v3_dunnhumby_nv_modulated_id_l2_1e4_seed43_v1'
assert screen.VERSION == 'nv-modulated-id-m2-l2-1e4-seed43-development-v1'
screen.read_source(REPORT, screen.configure(str(OUT)))
print('원본 결과 및 고정 소스 확인:', SOURCE_COMMIT)

## 1. 기존 결과·입력 확인 (학습 없음)
기존 M1/M2 선택규칙과 데이터·N/V 입력 해시를 대조합니다. 이전 checkpoint는 불러오지 않습니다. 기존 학습결과를 비교에 재사용할 뿐, 새 M2는 동일 초기화에서 공동학습합니다.

In [ ]:
cfg, prepared = screen.prepare(REPORT, OUT)
assert cfg.seeds == (43,) and cfg.epochs == 300 and cfg.pref_reg == .001
assert screen.spec()['role'] == 'M2' and not screen.spec()['weighted']
print('재사용:', [a['model_id'] for a in prepared['anchors']])
print('새 학습 1개:', screen.spec())
print('선택 규칙:', screen.selection(cfg))

## 2. M2 한 개 학습
완료 epoch마다 optimizer·난수상태를 Drive에 저장합니다. 중단 후 같은 노트북을 처음부터 재실행하면 저장 지점에서 이어지고, 완료 결과는 건너뜁니다. 자동 재연결은 보장하지 않습니다. 초기에 N/V 추천 gradient가 없거나 이후 표현이 수치0이면 진단을 남기고 멈춥니다.

In [ ]:
import torch
assert torch.cuda.is_available(), '전체 실험은 GPU 런타임을 사용하세요.'
paths = screen.run(cfg, prepared)
print(json.dumps(paths, ensure_ascii=False, indent=2))

## 3. 전체 절대지표·원본 ZIP
ZIP에는 absolute/comparison/same_epoch_comparison/curve/diagnostics CSV와 판독 JSON, 입력 진단, 새 M2의 초기·중간 probe가 들어갑니다. 같은 epoch 비교는 실제로 양쪽에 존재하는 평가점만 사용합니다. 전체·세그먼트와 진단을 함께 판독하며, 아래 불리언만으로 결론내리지 않습니다. 학습 중단 시에는 저장된 probe만 내려받을 수 있습니다.

In [ ]:
from zipfile import ZipFile, ZIP_DEFLATED
from google.colab import files
report_path = OUT/'reports/result.json'
paths = {}
if report_path.is_file():
    report = json.loads(report_path.read_text())
    assert report['code_version'] == screen.VERSION
    paths = report['paths']
    absolute = pd.read_csv(paths['absolute'])
    print('전체 절대지표 — 기존 M1, 기존 M2(.001), 새 M2(.0001)')
    print(absolute.set_index('model_id').T.to_string())
    print(json.dumps(report['reading'], ensure_ascii=False, indent=2))
    print('선택 checkpoint 비교:', paths['comparison'])
    print('동일 epoch 비교:', paths['same_epoch_comparison'])
    print('N/V 작동 진단:', paths['diagnostics'])
else:
    print('완료 결과 없음. 현재 저장된 작동 진단만 묶습니다. 성과 판정은 보류합니다.')
zip_path = Path('/content/nv_modulated_m2_l2_1e4_seed43_results.zip')
with ZipFile(zip_path, 'w', compression=ZIP_DEFLATED) as archive:
    for path in paths.values():
        archive.write(path, arcname=Path(path).name)
    if (OUT/'feature_diagnostic.json').is_file():
        archive.write(OUT/'feature_diagnostic.json', arcname='feature_diagnostic.json')
    for path in OUT.glob('arms/*/probe_epoch*.json'):
        archive.write(path, arcname=str(path.relative_to(OUT)))
files.download(str(zip_path))